# Image Classification using a CNN (Keras + TensorFlow)
**Dataset:** CIFAR-10 (60,000 color images, 32x32, 10 classes)

| Stage | Task |
|---|---|
| a | Loading and preprocessing the image data |
| b | Defining the model's architecture |
| c | Training the model |
| d | Estimating the model's performance |

```
Load data -> Preprocess -> Build CNN -> Train -> Evaluate
```

## Stage a: Loading and preprocessing the image data
- Load CIFAR-10: 50,000 train and 10,000 test images
- **Normalize** pixels 0-255 to 0-1
- Labels are integers (0-9), so we use `sparse_categorical_crossentropy` later (no one-hot needed)
- Show sample images to verify

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
%matplotlib inline

# Load data
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)

# Normalize
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Show sample images
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## Stage b: Defining the model's architecture
```
Input 32x32x3
 -> Conv(32) + ReLU -> MaxPool
 -> Conv(64) + ReLU -> MaxPool
 -> Conv(64) + ReLU
 -> Flatten
 -> Dense(64) + ReLU -> Dropout(0.5)
 -> Dense(10) + Softmax
```
- **Conv2D**: detects features (edges, shapes) using filters
- **MaxPooling**: shrinks the image, keeps important features
- **Flatten**: converts feature maps to 1D
- **Dropout**: randomly switches off neurons to reduce overfitting
- **Softmax**: probabilities for the 10 classes

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")
])

model.summary()

## Stage c: Training the model
- `optimizer="adam"`: adaptive optimizer, learns fast
- `loss="sparse_categorical_crossentropy"`: for integer labels
- `epochs=10`, `batch_size=64`
- `validation_data`: checks performance on test data after each epoch

In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history = model.fit(x_train, y_train,
                    epochs=10,
                    batch_size=64,
                    validation_data=(x_test, y_test))

## Stage d: Estimating the model's performance
1. Test loss and accuracy
2. Training vs validation curves
3. Classification report (precision, recall, F1)
4. Confusion matrix
5. Sample predictions

In [ ]:
# 1. Test loss and accuracy
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print("Test Loss     = %.3f" % test_loss)
print("Test Accuracy = %.3f" % test_acc)

In [ ]:
# 2. Accuracy and loss curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
# 3. Classification report
from sklearn.metrics import classification_report, confusion_matrix

y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# 4. Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks(range(10), class_names, rotation=45)
plt.yticks(range(10), class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Sample predictions (green = correct, red = wrong)
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i])
    color = "green" if y_pred[i] == y_test[i] else "red"
    plt.title("P: %s\nA: %s" % (class_names[y_pred[i]], class_names[y_test[i]]),
              color=color, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Conclusion
A CNN with 3 convolution layers, max pooling and dropout was trained on CIFAR-10. It reaches roughly **70% test accuracy** in 10 epochs. The confusion matrix shows the model mixes up similar classes (cat/dog, automobile/truck). Accuracy can be improved with data augmentation, more epochs, or a deeper network.